---
format: gfm
execute:
  enabled: false      # never execute on render; use stored outputs
---

## CarpeDIAMS

A package for deconvoluting dia/aif/msn data into pseudo-dda data. Builds on the foundation of corrdec and ms-dial but integrates new clustering methods of ms2 data and more sophisticated filtering methods for the deconvolution.

It is all about catching the DIA patterns in our MS data.

Three types of input can be used to generate an mgf of ms2 data:
* An MsExperiment object from xcms
* A mzml directory path and a target list of peak positions (mz and rt)
* mzml file(s) which are submitted to peak picking and then processed by the carpediams workflow


### How it works

As with corrdec and msdial the algorithm simply finds ms2 fragments that correlate with 

In [2]:
library(Spectra)
library(xcms)


This is xcms version 4.4.0 



Attaching package: ‘xcms’


The following object is masked from ‘package:Spectra’:

    ppm


The following object is masked from ‘package:stats’:

    sigma




In [111]:
devtools::load_all("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/modules/shared/carpeDIAMS")

ℹ Loading carpeDIAMS


In [4]:
library(tidyr)
library(dplyr)
library(purrr)
library(tibble)
library(readr)


Attaching package: ‘tidyr’


The following object is masked from ‘package:S4Vectors’:

    expand



Attaching package: ‘dplyr’


The following objects are masked from ‘package:xcms’:

    collect, groups


The following objects are masked from ‘package:S4Vectors’:

    first, intersect, rename, setdiff, setequal, union


The following objects are masked from ‘package:BiocGenerics’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
files <- list.files("/faststorage/project/forensics_TOFscreenings/BACKUP/TOF2_stoffer_compassxport", recursive = TRUE, full.names = TRUE)

tmp_folder <- "/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/results_all/drugs_annotation/tmp"

df <- tibble::tibble(
  file = files,
  sample_name = basename(files),
  file_size = file.size(files)
) |> 
mutate(
    peak_picked = paste0(tmp_folder, "/", gsub(".mzML$", "_peaks.rds", sample_name)),
    integration_ready = paste0(tmp_folder, "/", "integration_ready.rds"),
    integration1 = "#"
)

library(stringr)
TOF2_DB <- readr::read_csv("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/results_all/drugs_annotation/TOF-2_DB.csv")

COMPOUND_LIST <- readr::read_csv("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/results_all/drugs_annotation/compound_file_mapping.csv")

normalize <- function(x) {
  x |>
    tolower() |>
    str_replace_all("[\\-_\\. ]", "")
}

norm_db    <- normalize(TOF2_DB$name)
norm_files <- normalize(COMPOUND_LIST$basename)

# For each db name, find which filenames contain it
match_idx <- sapply(norm_db, function(db) {
  which(str_detect(norm_files, fixed(db)))
})

result <- data.frame(
  db_idx   = rep(seq_along(norm_db), lengths(match_idx)),
  file_idx = unlist(match_idx)
)

# Join in the actual columns
result <- data.frame(
  db_idx   = rep(seq_along(norm_db), lengths(match_idx)),
  file_idx = unlist(match_idx)
) |>
  mutate(
    TOF2_DB[db_idx, ],
    COMPOUND_LIST[file_idx, ]
  )

missing_frags <- result  |> 
  # mutate(
  #   `QI 1` = as.numeric(`QI 1`), 
  #   `QI 2` = as.numeric(`QI 2`), 
  #   `QI 3` = as.numeric(`QI 3`))  |> 
  select(`QI 1`, `QI 2`, `QI 3`) |> 
  map_dfc(is.na) |> 
  as.matrix() |> rowSums()

result <- result[missing_frags<3, ]



library(Spectra)
library(BiocParallel)
register(SerialParam())

mzml_path <- "/faststorage/project/forensics_TOFscreenings/BACKUP/TOF2_stoffer_compassxport/indkoering_forsoeg/indkoering_stoffer/gruppe_indkoering/"
origin_path <- "O:/HE_IFR-Rka/Instrumentdata/toks/UPLC-TOF/UPLC-TOF-2/indkoering_forsoeg/indkoering_stoffer/gruppe_indkoering/"
result_existing <- result |> mutate(mzml = gsub(origin_path, mzml_path, file), mzml = gsub(".d$", ".mzML", mzml))
result_existing <- result_existing |> filter(file.exists(mzml))

compounds <- result_existing$name |> unique()
files <- result_existing |> filter(name %in% compounds[1:5]) 


Rows: 951 Columns: 25
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (12): formula, name, CAS, QI 1, QI 2, QI 3, QI 1 min, comment3, comment4...
dbl (12): m/z, rt, comment 1, comment 2, minimum area, indivSigma, indivMass...
lgl  (1): relativeRetentiontime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2505 Columns: 3
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (3): compound, basename, file

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
msLevel(sps) <- as.integer((Spectra::scanIndex(sps) %% 2) + 1)

In [ ]:
process_mzml <- function(path, tmp_dir){
    sps <- Spectra(path, backend = MsBackendMzR()) |> Spectra::filterRt(c(30,1200))
    
    # Check if sps is annotated with ms2
    if (! (2 %in% unique(Spectra::msLevel(sps)))) {
        sps@backend@spectraData@listData$msLevel <- as.integer((Spectra::scanIndex(sps) %% 2) + 1)
    }

    

    cwp <- CentWaveParam(
        peakwidth = c(2,30),
        snthresh  = 6,
        ppm       = 12,
        prefilter = c(3, 1000),
        mzdiff    = 0.01)

    mse <- MsExperiment()
    spectra(mse) <- sps

    sampleData(mse) <- DataFrame(sample_name = unique(dataOrigin(sps)))   # MUST equal dataOrigin
    mse <- linkSampleData(mse, with = "sampleData.sample_name = spectra.dataOrigin")
    sps <- findChromPeaks(mse, cwp)

    fd <- chromPeaks(sps)
    rownames(fd) <- rownames(chromPeakData(sps))
    spectra_df <- spectra(sps)@backend@spectraData@listData |> bind_cols()

    fd_split <- fd |> as.data.frame() |> split(1:nrow(fd))
    names(fd_split) <- rownames(fd)

    scan_info <- extractScanInfo(feature_peaks = fd_split, filenames = fileNames(sps), spectra_df = spectra_df)
    spectra <- Ms2PseudoRaw(
        single_sample = scan_info,
        spectra_object = spectra(sps),
        maxDiff = 0.005,
        output_dir = tmp_dir,
        return_spectra = TRUE
    )

    features <- rownames(fd)
    cleaned_spectra <- clean_pseudo_spectra(
        targets        = features,
        file_directory = tmp_dir,
        fd             = as.data.frame(fd),
        ms1_ppm        = 10,
        grouper        = ppm_upgma(ppm = 30),
        buffer         = 50L,
        fdr_thresh     = 0.1,
        cor_thresh     = 0.1,
        min_count      = 10L,
        report         = FALSE,
        mgf_dir        = tmp_dir,
        progress       = TRUE
        )

    export_mgf(
        results = cleaned_spectra, 
        fd = fd, 
        mgf_path = paste0(tmp_dir, "/all_features.mgf"), 
        ion_mode = "POSITIVE"
        )
    cli::cli_process_done("YO, everything's finished!")
    return(cleaned_spectra)
}


In [ ]:
devtools::load_all("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/modules/shared/carpeDIAMS")
process_mzml(path = files$mzml[1], tmp_dir = tmp_dir)

ℹ Loading carpeDIAMS


── Processing 17-alpha-trenbolon_10ul_1-A,5_01_40098.mzML ──────────────────────────────────────────────────────────────

ℹ Loading spectra and filtering RT to 30-1200s

! No MS2 annotation found - assigning by frame parity (bbCID).

ℹ Loading spectra and filtering RT to 30-1200s

ℹ Assigned MS1/MS2 by parity: 2836 / 2836 scans.

ℹ Loading spectra and filtering RT to 30-1200s
✔ Loading spectra and filtering RT to 30-1200s [5.8s]


ℹ Detecting chromatographic peaks (centWave)

[----------------------------------------------------------] 0/1 (  0%) in  0s

[==========================================================] 1/1 (100%) in 24s



ℹ Found 1109 chromatographic peaks.

ℹ Detecting chromatographic peaks (centWave)
✔ Detecting chromatographic peaks (centWave) [25s]


ℹ Extracting scan info and building pseudo-raw pool
Finished Sample 1

✔ Extracting scan info and building pseudo-raw pool [18.4s]


ℹ Building and filtering pseudo-spectra (1109 features)


In [75]:
sps <- Spectra(files$mzml[1], backend = MsBackendMzR()) |> Spectra::filterRt(c(30,1200))
sps@backend@spectraData@listData$msLevel <- as.integer((Spectra::scanIndex(sps) %% 2) + 1)

In [76]:
library(MsExperiment); library(xcms); library(MsExperiment); library(Spectra)

cwp        <- CentWaveParam(
  peakwidth = c(2,30),
  snthresh  = 6,
  ppm       = 12,
  prefilter = c(3, 1000),
  mzdiff    = 0.01)

mse <- MsExperiment()
spectra(mse) <- sps

sampleData(mse) <- DataFrame(sample_name = unique(dataOrigin(sps)))   # MUST equal dataOrigin
mse <- linkSampleData(mse, with = "sampleData.sample_name = spectra.dataOrigin")
xdata <- findChromPeaks(mse, cwp)


[----------------------------------------------------------] 0/1 (  0%) in  0s

[==========================================================] 1/1 (100%) in 24s




In [77]:
xdata

Object of class XcmsExperiment 
 Spectra: MS1 (2836) MS2 (2836) 
 Experiment data: 1 sample(s)
 Sample data links:
  - spectra: 1 sample(s) to 5672 element(s).
 xcms results:
  - chromatographic peaks: 1109 in MS level(s): 1 

In [78]:
library(data.table)

fd <- chromPeaks(xdata)
rownames(fd) <- rownames(chromPeakData(xdata))
spectra_df <- spectra(xdata)@backend@spectraData@listData |> bind_cols()

In [99]:
tmp_dir <- "/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/modules/shared/carpeDIAMS/tmp"
devtools::load_all("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/modules/shared/carpeDIAMS")

ℹ Loading carpeDIAMS


In [80]:
a <- fd |> as.data.frame() |> split(1:nrow(fd))
names(a) <- rownames(fd)
scan_info <- extractScanInfo(feature_peaks = a, filenames = fileNames(xdata), spectra_df = spectra_df)
spectra <- Ms2PseudoRaw(
    single_sample = scan_info,
    spectra_object = spectra(xdata),
    maxDiff = 0.005,
    output_dir = tmp_dir,
    return_spectra = TRUE
)

Finished Sample 1



In [102]:
features <- rownames(fd)

a <- clean_pseudo_spectra(
  targets        = features,
  file_directory = tmp_dir,
  fd             = as.data.frame(fd),
  ms1_ppm        = 10,
  grouper        = ppm_upgma(ppm = 30),
  buffer         = 50L,
  fdr_thresh     = 0.1,
  cor_thresh     = 0.1,
  min_count      = 10L,
  report         = FALSE,
  mgf_dir        = tmp_dir,
  progress       = TRUE
)


  [1109/1109] CP1109         


In [101]:
export_mgf(results = a, fd = fd, mgf_path = paste0(tmp_dir, "/all_features.mgf"), ion_mode = "POSITIVE")

In [ ]:
a = get_raw_spec(feature_id = "CP2341",file_directory = tmp_dir)
print(a)

       sample feature       mz fragment_mz   int ms1_scan ms2_scan ms1_int
        <num>  <char>    <num>       <num> <num>    <int>    <int>   <num>
    1:      1  CP2341 61.02926    44.93030   404        2        3     235
    2:      1  CP2341 61.02926    44.94308   272        2        3     235
    3:      1  CP2341 61.02926    44.94879   260        2        3     235
    4:      1  CP2341 61.02926    44.96141   248        2        3     235
    5:      1  CP2341 61.02926    44.97400   236        2        3     235
   ---                                                                    
11990:      1  CP2341 61.02926    61.04062  2400      120      121     696
11991:      1  CP2341 61.02926    61.88616   272      120      121     696
11992:      1  CP2341 61.02926    61.92665   296      120      121     696
11993:      1  CP2341 61.02926    62.37306    88      120      121     696
11994:      1  CP2341 61.02926    62.99213   412      120      121     696
       feature_id
       

In [44]:
targets = "CP2341"
file_directory = tmp_dir
fd               = as.data.frame(fd)
con              = NULL
ms1_ppm          = 10
grouper          = ppm_upgma(ppm = 30)
buffer           = 50L
fdr_count        = 30L
cor_count        = 100L
min_count        = 10L
mgf_dir          = NULL
fetch_chunk_size = 200L
progress         = TRUE
report = FALSE

own_con <- is.null(con)
if (own_con) {
  con <- setup_connection(file_directory)
  #on.exit(DBI::dbDisconnect(con, shutdown = TRUE), add = TRUE)
}

fd_df <- NULL
if (!is.null(fd)) {
  fd_df <- as.data.frame(fd)
  if ("feature_id" %in% colnames(fd_df)) rownames(fd_df) <- fd_df$feature_id
  if (!"mzmed" %in% colnames(fd_df) && "mz" %in% colnames(fd_df))
    fd_df$mzmed <- fd_df$mz
  if (!"rtmed" %in% colnames(fd_df) && "rt" %in% colnames(fd_df))
    fd_df$rtmed <- fd_df$rt
}
has_fd <- !is.null(fd_df)

total <- length(targets)
done  <- 0L
fetch_chunks <- split(targets, ceiling(seq_along(targets) / fetch_chunk_size))

results <- data.table::rbindlist(lapply(fetch_chunks, function(chunk_ids) {
  raw <- batch_get_feature_data(chunk_ids, con, chunk_size = length(chunk_ids))
  if (nrow(raw) == 0) {
    done <<- done + length(chunk_ids)
    return(NULL)
  }
  by_feature <- split(raw, by = "feature")

  data.table::rbindlist(lapply(chunk_ids, function(target) {
    done <<- done + 1L
    if (progress) cat(sprintf("\r  [%d/%d] %s         ", done, total, target))

    dt <- by_feature[[target]]
    if (is.null(dt) || nrow(dt) == 0) return(NULL)
    print(dt)

    mz_target <- if (has_fd && target %in% rownames(fd_df)) fd_df[target, "mzmed"]
                  else dt$mz[1]
    print(mz_target)
    if (length(mz_target) != 1 || is.na(mz_target)) return(NULL)
    rt_target <- if (has_fd && target %in% rownames(fd_df)) fd_df[target, "rtmed"]
                  else NA_real_

    print(rt_target)
    out <- tryCatch(
      process_pseudo_spec(dt, mz_target = mz_target,
                          grouper = grouper, buffer = buffer,
                          min_count = min_count),
      error = function(e) NULL
    )
    if (is.null(out) || nrow(out$correlations) == 0) return(NULL)

    pseudo_spec <- select_pseudo_spec(out$correlations, mz_target,
                                      ms1_ppm, cor_count, fdr_count)
    if (nrow(pseudo_spec) == 0) return(NULL)

    if (report) {
      plots <- Filter(Negate(is.null),
                      .pseudo_spec_plots(out$grouped_signal, out$correlations,
                                          pseudo_spec, target))
      if (length(plots)) {
        if (requireNamespace("cowplot", quietly = TRUE)) {
          print(cowplot::plot_grid(plotlist = plots, ncol = 2))
        } else {
          for (p in plots) print(p)
        }
      }
    }

    if (!is.null(mgf_dir)) {
      write_simple_mgf(
        pseudo_spec, mz_target = mz_target, feature_id = target,
        path = file.path(mgf_dir, paste0(target, ".mgf")),
        rt = rt_target
      )
    }

    n_samples_per_group <- out$grouped_signal[
      group %in% pseudo_spec$group,
      .(n_samples = data.table::uniqueN(sample)),
      by = group
    ]
    pseudo_spec <- merge(pseudo_spec, n_samples_per_group,
                          by = "group", all.x = TRUE)

    data.table::data.table(
      mz_g2           = pseudo_spec$fragment_mz,
      grouped_mz      = as.integer(pseudo_spec$group),
      mz              = pseudo_spec$fragment_mz,
      total_intensity = pseudo_spec$int,
      n_samples       = as.integer(pseudo_spec$n_samples),
      count           = as.integer(pseudo_spec$count),
      r               = pseudo_spec$correlation,
      t_stat          = pseudo_spec$correlation * sqrt(pseudo_spec$count - 2) /
         sqrt(1 - pseudo_spec$correlation^2),
      p_value         = pseudo_spec$p_value,
      fdr             = pseudo_spec$fdr,
      feature_id      = target
    )
  }), fill = TRUE)
}), fill = TRUE)

if (progress) cat("\n")
results

DBI::dbDisconnect(con, shutdown = TRUE)

  [1/1] CP2341                sample feature       mz fragment_mz   int ms1_scan ms2_scan ms1_int
        <num>  <char>    <num>       <num> <num>    <int>    <int>   <num>
    1:      1  CP2341 61.02926    44.93030   404        2        3     235
    2:      1  CP2341 61.02926    44.94308   272        2        3     235
    3:      1  CP2341 61.02926    44.94879   260        2        3     235
    4:      1  CP2341 61.02926    44.96141   248        2        3     235
    5:      1  CP2341 61.02926    44.97400   236        2        3     235
   ---                                                                    
11990:      1  CP2341 61.02926    61.04062  2400      120      121     696
11991:      1  CP2341 61.02926    61.88616   272      120      121     696
11992:      1  CP2341 61.02926    61.92665   296      120      121     696
11993:      1  CP2341 61.02926    62.37306    88      120      121     696
11994:      1  CP2341 61.02926    62.99213   412      120      121     696
[1

mz_g2,grouped_mz,mz,total_intensity,n_samples,count,r,t_stat,p_value,fdr,feature_id
<dbl>,<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
55.93444,1038,55.93444,728.2353,1,17,0.25645640,1.02761926,3.204158e-01,0.4078019266,CP2341
59.00803,1313,59.00803,4361.6000,1,15,0.07387197,0.26707890,7.935984e-01,0.8546443952,CP2341
61.04109,1495,61.04109,2560.0000,1,16,0.64209890,3.13389735,7.322272e-03,0.0205023628,CP2341
63.00278,1658,63.00278,569.0000,1,16,0.29646812,1.16149998,2.648593e-01,0.3708030627,CP2341
57.93610,2904,57.93610,455.4667,1,15,0.40586174,1.60115995,1.333515e-01,0.2333651704,CP2341
55.93553,7700,55.93553,581.2632,1,19,0.43190587,1.97445018,6.480432e-02,0.1296086420,CP2341
54.98607,9293,54.98607,956.3333,1,12,0.84575480,5.01246823,5.275548e-04,0.0022292681,CP2341
60.98725,10033,60.98725,17842.4348,1,23,0.30492183,1.46719923,1.571394e-01,0.2444390432,CP2341
57.93479,10059,57.93479,456.3636,1,11,0.86234605,5.10949871,6.369337e-04,0.0022292681,CP2341


In [36]:
print(results)

Null data.table (0 rows and 0 cols)


In [ ]:
b <- sps |>  |> 
    peaksData()

n_peaks  <- vapply(b, nrow, integer(1))
pks_matrix <- do.call(rbind, b)
print(sum(pks_matrix[, 2]))

b <- sps |> 
    Spectra::filterAcquisitionNum() |> 
    peaksData()

n_peaks  <- vapply(b, nrow, integer(1))
pks_matrix <- do.call(rbind, b)
print(sum(pks_matrix[, 2]))


[1] 6467674105
[1] 5135526144
